In [1]:
!pip install mplsoccer


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplsoccer as ms
from mplsoccer import Sbopen
from collections import defaultdict, Counter
import random

# loading games

In [3]:
results_raw = pd.read_csv('data/results.csv')
former_names = pd.read_csv('data/former_names.csv')

results_raw['date'] = pd.to_datetime(results_raw['date'])
results_raw = results_raw.dropna(subset=['home_score', 'away_score'])
results_raw['home_score'] = results_raw['home_score'].astype(int)
results_raw['away_score'] = results_raw['away_score'].astype(int)

print(f"Total matches loaded: {len(results_raw)}")
print(f"Date range: {results_raw['date'].min().date()} → {results_raw['date'].max().date()}")
print(f"\nSample:\n{results_raw.tail(3)}")

Total matches loaded: 49365
Date range: 1872-11-30 → 2026-06-06

Sample:
            date  home_team      away_team  ...    city      country neutral
49362 2026-06-06    Myanmar           Guam  ...  Manila  Philippines    True
49363 2026-06-06  Lithuania         Latvia  ...  Kaunas    Lithuania   False
49364 2026-06-06    Estonia  Faroe Islands  ...   Pärnu      Estonia   False

[3 rows x 9 columns]


# team name normalisation

In [4]:
# team name normalisation
# two sources of renames:
# 1. former_names.csv — historical name changes (e.g. Zaire → DR Congo)
# 2. manual map — dataset inconsistencies and FIFA vs common naming differences
former_names = pd.read_csv('data/former_names.csv')

# building rename dict from former_names.csv automatically
# every "former" name → maps to "current" name
FORMER_NAME_MAP = dict(zip(former_names['former'], former_names['current']))

# manual overrides — dataset-specific inconsistencies, FIFA naming vs common naming,
# and anything NOT covered by former_names.csv
MANUAL_MAP = {
    # USA
    "United States":                "USA",
    "United States of America":     "USA",

    # Korea
    "Korea Republic":               "South Korea",
    "Korea DPR":                    "North Korea",

    # Iran
    "IR Iran":                      "Iran",

    # China
    "China PR":                     "China",

    # Cabo Verde — FIFA renamed from "Cape Verde" to "Cabo Verde" in 2014,
    # but the dataset uses "Cape Verde" throughout (all 238 matches).
    # If your group_fixtures.csv uses "Cabo Verde", we map it back to match the dataset.
    "Cabo Verde":                   "Cape Verde",

    # Ivory Coast — dataset uses French name
    "Côte d'Ivoire":                "Ivory Coast",
    "Cote d'Ivoire":                "Ivory Coast",

    # Congo variants not in former_names.csv
    "Congo DR":                     "DR Congo",
    "Congo, DR":                    "DR Congo",
    "Democratic Republic of the Congo": "DR Congo",

    # Bosnia
    "Bosnia-Herzegovina":           "Bosnia & Herzegovina",
    "Bosnia and Herzegovina":       "Bosnia & Herzegovina",

    # Trinidad
    "Trinidad and Tobago":          "Trinidad & Tobago",

    # São Tomé
    "São Tomé and Príncipe":        "São Tomé & Príncipe",
    "Sao Tome and Principe":        "São Tomé & Príncipe",

    # Kyrgyzstan
    "Kyrgyz Republic":              "Kyrgyzstan",

    # North Macedonia — also covered by former_names.csv but adding for safety
    "Macedonia":                    "North Macedonia",

    # Republic of Ireland — dataset uses full name, some sources say just "Ireland"
    "Ireland":                      "Republic of Ireland",

    # Eswatini — also in former_names.csv but FIFA sources sometimes still use old name
    "Swaziland":                    "Eswatini",

    # Czech Republic / Czechia
    "Czechia":                      "Czech Republic",
}

# merging both maps — manual overrides take priority over former_names.csv
NAME_MAP = {**FORMER_NAME_MAP, **MANUAL_MAP}

def normalise_name(name):
    return NAME_MAP.get(name, name)

# applying to results dataframe
results_raw['home_team'] = results_raw['home_team'].apply(normalise_name)
results_raw['away_team'] = results_raw['away_team'].apply(normalise_name)

# filtering matches, tournaments, and assigning weights

In [5]:
group_fixtures = pd.read_csv('data/group_fixtures.csv')
group_fixtures['home_team'] = group_fixtures['home_team'].apply(normalise_name)
group_fixtures['away_team'] = group_fixtures['away_team'].apply(normalise_name)

wc_teams = set(group_fixtures['home_team'].tolist() + group_fixtures['away_team'].tolist())

print(f"WC 2026 teams ({len(wc_teams)}):", sorted(wc_teams))
print(f"Total WC teams defined: {len(wc_teams)}")

WC 2026 teams (48): ['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia & Herzegovina', 'Brazil', 'Canada', 'Cape Verde', 'Colombia', 'Croatia', 'Curaçao', 'Czech Republic', 'DR Congo', 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran', 'Iraq', 'Ivory Coast', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Turkey', 'USA', 'Uruguay', 'Uzbekistan']
Total WC teams defined: 48


In [6]:
# filtering to relevant matches (post-2010) and assigning tournament weights

CUTOFF_DATE = pd.Timestamp('2010-01-01')
results = results_raw[results_raw['date'] >= CUTOFF_DATE].copy()

# tournament importance weights (higher = more predictive signal)
TOURNAMENT_WEIGHTS = {
    # global
    'FIFA World Cup':                        1.00,
    'FIFA World Cup qualification':          0.80,
    'Confederations Cup':                    0.85,
    'FIFA Series':                           0.70,

    # europe
    'UEFA Euro':                             0.90,
    'UEFA Euro qualification':               0.75,
    'UEFA Nations League':                   0.75,

    # south america
    'Copa América':                          0.90,
    'Copa América qualification':            0.75,
    'Superclásico de las Américas':          0.70,
    'CONMEBOL–UEFA Cup of Champions':        0.70,

    # africa
    'African Cup of Nations':                0.85,
    'African Cup of Nations qualification':  0.70,
    'Arab Cup':                              0.60,
    'Arab Cup qualification':                0.50,
    'CECAFA Cup':                            0.45,
    'COSAFA Cup':                            0.45,

    # asia
    'AFC Asian Cup':                         0.85,
    'AFC Asian Cup qualification':           0.70,
    'AFC Challenge Cup':                     0.60,
    'AFC Challenge Cup qualification':       0.50,
    'WAFF Championship':                     0.55,
    'EAFF Championship':                     0.55,
    'EAFF Championship qualification':       0.50,
    'Gulf Cup':                              0.55,
    'SAFF Cup':                              0.50,
    'CAFA Nations Cup':                      0.50,
    'AFF Championship':                      0.55,
    'AFF Championship qualification':        0.50,
    'ASEAN Championship':                    0.55,
    'ASEAN Championship qualification':      0.50,

    # north/central america & caribbean
    'Gold Cup':                              0.80,
    'Gold Cup qualification':               0.65,
    'CONCACAF Nations League':               0.75,
    'CONCACAF Nations League qualification': 0.60,
    'CONCACAF Series':                       0.65,
    'CFU Caribbean Cup':                     0.55,
    'CFU Caribbean Cup qualification':       0.45,
    'UNCAF Cup':                             0.50,

    # oceania
    'Oceania Nations Cup':                   0.75,
    'Oceania Nations Cup qualification':     0.60,
}

def get_weight(tournament):
    for key, w in TOURNAMENT_WEIGHTS.items():
        if key.lower() in tournament.lower():
            return w
    return 0.4  # default for unrecognised tournaments

results['tournament_weight'] = results['tournament'].apply(get_weight)

In [7]:
# recency weight: exponential decay, half-life ~3 years
# matches from today feel 1.0; matches from 3 years ago feel ~0.5
TODAY = pd.Timestamp.today()
HALF_LIFE_DAYS = 3 * 365
results['days_ago'] = (TODAY - results['date']).dt.days
results['recency_weight'] = np.exp(-np.log(2) * results['days_ago'] / HALF_LIFE_DAYS)

# combined sample weight
results['weight'] = results['tournament_weight'] * results['recency_weight']

print(f"Matches after 2010 cutoff: {len(results)}")
print(f"\nWeight distribution:\n{results['weight'].describe().round(3)}")

Matches after 2010 cutoff: 15777

Weight distribution:
count    15777.000
mean         0.186
std          0.205
min          0.009
25%          0.034
50%          0.093
75%          0.293
max          0.945
Name: weight, dtype: float64


In [8]:
# rolling form features (last 10 matches per team, exponentially weighted)

def get_team_matches(df, team):
    """Return all matches involving a team, with their perspective (home/away)."""
    home = df[df['home_team'] == team].copy()
    home['team'] = team
    home['opponent'] = home['away_team']
    home['goals_for'] = home['home_score']
    home['goals_against'] = home['away_score']
    home['is_home'] = True
    home['result'] = np.where(home['home_score'] > home['away_score'], 'W',
                     np.where(home['home_score'] < home['away_score'], 'L', 'D'))

    away = df[df['away_team'] == team].copy()
    away['team'] = team
    away['opponent'] = away['home_team']
    away['goals_for'] = away['away_score']
    away['goals_against'] = away['home_score']
    away['is_home'] = False
    away['result'] = np.where(away['away_score'] > away['home_score'], 'W',
                     np.where(away['away_score'] < away['home_score'], 'L', 'D'))

    cols = ['date', 'team', 'opponent', 'goals_for', 'goals_against',
            'result', 'is_home', 'tournament', 'tournament_weight',
            'recency_weight', 'weight', 'neutral']
    combined = pd.concat([home[cols], away[cols]]).sort_values('date').reset_index(drop=True)
    return combined

# rolling form features (last 10 matches)

In [9]:
def rolling_form(team_matches, n=10):
    """
    Compute exponentially weighted rolling form over last n matches.
    Returns a single-row summary dict.
    """
    recent = team_matches.tail(n).copy()
    if len(recent) == 0:
        return {}

    w = recent['weight'].values
    w = w / w.sum()  # normalise

    gf = recent['goals_for'].values
    ga = recent['goals_against'].values
    pts = np.where(recent['result'] == 'W', 3,
          np.where(recent['result'] == 'D', 1, 0))

    return {
        'avg_goals_for':     float(np.average(gf, weights=w)),
        'avg_goals_against': float(np.average(ga, weights=w)),
        'avg_points':        float(np.average(pts, weights=w)),
        'win_rate':          float(np.average(recent['result'] == 'W', weights=w)),
        'draw_rate':         float(np.average(recent['result'] == 'D', weights=w)),
        'loss_rate':         float(np.average(recent['result'] == 'L', weights=w)),
        'avg_goal_diff':     float(np.average(gf - ga, weights=w)),
        'clean_sheet_rate':  float(np.average(ga == 0, weights=w)),
        'failed_to_score_rate': float(np.average(gf == 0, weights=w)),
        'n_matches_available':  len(team_matches),
    }

In [10]:
# building form table for all WC teams
form_records = []
for team in sorted(wc_teams):
    tm = get_team_matches(results, team)
    form = rolling_form(tm, n=10)
    if form:
        form['team'] = team
        form_records.append(form)
    else:
        print(f"  WARNING: No data found for '{team}'")

form_df = pd.DataFrame(form_records).set_index('team')

# incorporating elo

In [11]:
# elo ratings

ELO_RATINGS = {
    "France":          2062,
    "Spain":           2155,
    "England":         2021,
    "Brazil":          1991,
    "Argentina":       2114,
    "Portugal":        1986,
    "Netherlands":     1944,
    "Belgium":         1893,
    "Germany":         1932,
    "Colombia":        1977,
    "Croatia":         1908,
    "Morocco":         1824,
    "USA":             1726,
    "Mexico":          1875,
    "Uruguay":         1892,
    "Senegal":         1867,
    "Japan":           1906,
    "Iran":            1772,
    "Switzerland":     1891,
    "Ecuador":         1935,
    "Canada":          1788,
    "Australia":       1777,
    "South Korea":     1758,
    "Scotland":        1782,
    "Tunisia":         1628,
    "Egypt":           1696,
    "Paraguay":        1833,
    "Panama":          1730,
    "Saudi Arabia":    1569,
    "Czech Republic":  1740,
    "Qatar":           1421,
    "New Zealand":     1562,
    "DR Congo":        1661,
    'Bosnia & Herzegovina': 1595,
    'Sweden':          1712,
    'Austria':         1830,
    'Norway':          1917,
    'Turkey':          1911,
    'Jordan':          1685,
    'Uzbekistan':      1718,
    'Iraq':            1618,
    'Algeria':         1760,
    'Cape Verde':      1578,
    'Ghana':           1510,
    'Ivory Coast':     1695,
    'South Africa':    1528,
    'Haiti':           1548,
    'Curaçao':         1434
}

# the values above are as of June 7, 2026.

elo_df = pd.DataFrame.from_dict(ELO_RATINGS, orient='index', columns=['elo'])
elo_df.index.name = 'team'

print(f"Elo ratings loaded for {len(elo_df)} teams")
print(elo_df.sort_values('elo', ascending=False).head(10))

Elo ratings loaded for 48 teams
              elo
team             
Spain        2155
Argentina    2114
France       2062
England      2021
Brazil       1991
Portugal     1986
Colombia     1977
Netherlands  1944
Ecuador      1935
Germany      1932


# incorporating sqaud value

In [12]:
# squad market values (INR Cr, from Transfermarkt June 2026)
SQUAD_VALUE = pd.read_csv('data/squad_value.csv')
SQUAD_VALUE['Squad'] = SQUAD_VALUE['Squad'].apply(normalise_name)
SQUAD_VALUE.set_index('Squad', inplace=True)

In [13]:
SQUAD_VALUE['Market Value_Numeric'] = (
    SQUAD_VALUE['Market Value']
    .str.replace('₹', '')
    .str.replace(' Cr', '')
    .str.replace(',', '')
    .astype(float))

GLOBAL_AVG_SQUAD = SQUAD_VALUE['Market Value_Numeric'].mean()

In [14]:
def get_squad_value(team):
    """
    Returns the numeric market value for a team.
    If the team is not found, returns the global average squad value.
    """
    if team in SQUAD_VALUE.index:
        return SQUAD_VALUE.loc[team, 'Market Value_Numeric']
    return GLOBAL_AVG_SQUAD

# final master table (form + elo + squad value) for all 48 teams

In [15]:
# building master features table

# merging form + elo
master = form_df.join(elo_df, how='left')
master['squad_value'] = master.index.map(get_squad_value)

# flagging any missing Elo
missing_elo = master[master['elo'].isna()]
missing_squad = master[master['squad_value'].isna()]
if len(missing_elo):
    print(f"WARNING: Missing Elo for: {missing_elo.index.tolist()}")
if len(missing_squad):
    print(f"WARNING: Missing squad value for: {missing_squad.index.tolist()}")

# elo z-score (useful as a model feature)
master['elo_zscore'] = (master['elo'] - master['elo'].mean()) / master['elo'].std()

print(f"\nMaster features table: {master.shape}")
print(master.round(3))


Master features table: (48, 13)
                      avg_goals_for  avg_goals_against  ...  squad_value  elo_zscore
team                                                    ...                         
Algeria                       1.779              0.469  ...     2055.000      -0.140
Argentina                     1.954              0.369  ...     6260.000       1.891
Australia                     1.537              0.929  ...      620.000      -0.043
Austria                       2.756              0.416  ...     1938.000       0.261
Belgium                       3.816              0.557  ...     4380.000       0.623
Bosnia & Herzegovina          2.013              1.063  ...     1213.000      -1.087
Brazil                        2.308              0.986  ...     7426.000       1.185
Canada                        1.106              0.416  ...     1573.000       0.020
Cape Verde                    1.800              1.197  ...      436.000      -1.185
Colombia                      2.

In [16]:
# tournaments considered competitive and relevant for training

RELEVANT_TOURNAMENTS = [
    'FIFA World Cup',
    'FIFA World Cup qualification',
    'Confederations Cup',
    'FIFA Series',

    'UEFA Euro',
    'UEFA Euro qualification',
    'UEFA Nations League',

    'Copa América',
    'Copa América qualification',
    'Superclásico de las Américas',
    'CONMEBOL–UEFA Cup of Champions',

    'African Cup of Nations',
    'African Cup of Nations qualification',
    'CECAFA Cup',
    'COSAFA Cup',
    'Arab Cup',
    'Arab Cup qualification',

    'AFC Asian Cup',
    'AFC Asian Cup qualification',
    'AFC Challenge Cup',
    'AFC Challenge Cup qualification',
    'WAFF Championship',
    'EAFF Championship',
    'EAFF Championship qualification',
    'Gulf Cup',
    'SAFF Cup',
    'CAFA Nations Cup',
    'AFF Championship',
    'AFF Championship qualification',
    'ASEAN Championship',
    'ASEAN Championship qualification',

    'Gold Cup',
    'Gold Cup qualification',
    'CONCACAF Nations League',
    'CONCACAF Nations League qualification',
    'CONCACAF Series',
    'CFU Caribbean Cup',
    'CFU Caribbean Cup qualification',
    'UNCAF Cup',

    'Oceania Nations Cup',
    'Oceania Nations Cup qualification'
]

def is_relevant(tournament):
    t = tournament.lower()
    return any(r.lower() in t for r in RELEVANT_TOURNAMENTS)

# building the prediction model

In [17]:
GLOBAL_AVG_ELO = 1400  # roughly the middle of the international Elo scale

def get_elo(team, elo_dict):
    return elo_dict.get(team, GLOBAL_AVG_ELO)

In [18]:
def compute_team_features_at_date(results_df, team, before_date, n=10):
    """
    Compute rolling form features for a team using only matches
    played strictly before `before_date`. Prevents data leakage.
    """
    tm = get_team_matches(results_df, team)
    tm = tm[tm['date'] < before_date].tail(n)

    if len(tm) < 3:  # not enough history, skip this match
        return None

    w = tm['weight'].values
    w = w / w.sum()

    gf  = tm['goals_for'].values
    ga  = tm['goals_against'].values
    pts = np.where(tm['result'] == 'W', 3,
          np.where(tm['result'] == 'D', 1, 0))

    return {
        'avg_goals_for':      float(np.average(gf, weights=w)),
        'avg_goals_against':  float(np.average(ga, weights=w)),
        'avg_points':         float(np.average(pts, weights=w)),
        'win_rate':           float(np.average(tm['result'] == 'W', weights=w)),
        'draw_rate':          float(np.average(tm['result'] == 'D', weights=w)),
        'avg_goal_diff':      float(np.average(gf - ga, weights=w)),
        'clean_sheet_rate':   float(np.average(ga == 0, weights=w)),
        'n_matches':          len(tm),
    }

In [19]:
def build_training_data(results_df, elo_dict, relevant_only=True):
    """
    Build a training dataset of matches with time-safe features.
    Uses fallback Elo for teams not in the WC pool.
    """
    df = results_df.copy()
    if relevant_only:
        df = df[df['tournament'].apply(is_relevant)]

    rows = []
    skipped = 0

    for _, row in df.iterrows():
        ht, at = row['home_team'], row['away_team']
        date   = row['date']

        hf = compute_team_features_at_date(results_df, ht, date)
        af = compute_team_features_at_date(results_df, at, date)

        # skip only if insufficient match history (< 3 matches)
        if hf is None or af is None:
            skipped += 1
            continue

        elo_h = get_elo(ht, elo_dict)
        elo_a = get_elo(at, elo_dict)

        hg = int(row['home_score'])
        ag = int(row['away_score'])

        feat = {
            'result':       1 if hg > ag else (-1 if hg < ag else 0),
            'elo_diff':     elo_h - elo_a,
            'elo_home':     elo_h,
            'elo_away':     elo_a,
            'home_score':     hg,
            'away_score':     ag,
            'gf_diff':      hf['avg_goals_for']    - af['avg_goals_for'],
            'ga_diff':      hf['avg_goals_against'] - af['avg_goals_against'],
            'pts_diff':     hf['avg_points']        - af['avg_points'],
            'gd_diff':      hf['avg_goal_diff']     - af['avg_goal_diff'],
            'wr_diff':      hf['win_rate']          - af['win_rate'],
            'cs_diff':      hf['clean_sheet_rate']  - af['clean_sheet_rate'],
            'home_avg_gf':  hf['avg_goals_for'],
            'home_avg_ga':  hf['avg_goals_against'],
            'home_avg_pts': hf['avg_points'],
            'home_win_rate':hf['win_rate'],
            'home_n':       hf['n_matches'],
            'away_avg_gf':  af['avg_goals_for'],
            'away_avg_ga':  af['avg_goals_against'],
            'away_avg_pts': af['avg_points'],
            'away_win_rate':af['win_rate'],
            'away_n':       af['n_matches'],
            'is_neutral':   int(str(row['neutral']).upper() == 'TRUE'),
            'is_wc':        int('World Cup' in str(row['tournament'])),
            'is_knockout':  int(any(k in str(row['tournament']).lower()
                                    for k in ['final', 'semifinal', 'quarter'])),
            'squad_value_home': get_squad_value(ht),
            'squad_value_away': get_squad_value(at),
            'squad_value_ratio': np.log(max(get_squad_value(ht), 1) / max(get_squad_value(at), 1)),
            'weight':       row['weight']}
        rows.append(feat)

    print(f"Built {len(rows)} training rows (skipped {skipped})")
    return pd.DataFrame(rows)

print("Building time-aware training data...")
train_df = build_training_data(results, ELO_RATINGS)
print(train_df['result'].value_counts())

Building time-aware training data...
Built 9629 training rows (skipped 174)
result
 1    4556
-1    2937
 0    2136
Name: count, dtype: int64


## Training XGBoost

In [20]:
# XGBOOST GOAL REGRESSORS

from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

FEATURE_COLS = [
    'elo_diff', 'elo_home', 'elo_away',
    'gf_diff', 'ga_diff', 'pts_diff', 'gd_diff', 'wr_diff', 'cs_diff',
    'home_avg_gf', 'home_avg_ga', 'home_avg_pts', 'home_win_rate',
    'away_avg_gf', 'away_avg_ga', 'away_avg_pts', 'away_win_rate',
    'is_neutral', 'is_wc', 'is_knockout', 'squad_value_home', 'squad_value_away',
    'squad_value_ratio']

X      = train_df[FEATURE_COLS].values
y_home = train_df['home_score'].values
y_away = train_df['away_score'].values
w      = train_df['weight'].values

# cross-validating both regressors
tscv = TimeSeriesSplit(n_splits=5)
home_maes, away_maes = [], []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_tr, X_val   = X[train_idx], X[val_idx]
    yh_tr, yh_val = y_home[train_idx], y_home[val_idx]
    ya_tr, ya_val = y_away[train_idx], y_away[val_idx]
    w_tr          = w[train_idx]

    home_model = XGBRegressor(
        n_estimators=250, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbosity=0
    )
    away_model = XGBRegressor(
        n_estimators=250, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbosity=0
    )

    home_model.fit(X_tr, yh_tr, sample_weight=w_tr)
    away_model.fit(X_tr, ya_tr, sample_weight=w_tr)

    home_mae = mean_absolute_error(yh_val, home_model.predict(X_val))
    away_mae = mean_absolute_error(ya_val, away_model.predict(X_val))
    home_maes.append(home_mae)
    away_maes.append(away_mae)

In [21]:
WC_HOST_NATIONS = {"USA", "Canada", "Mexico"}
home_advantage = 0.25

In [22]:
ELO_SCALE_K = 0.001   # higher = more extreme scores

def elo_scale_lambdas(exp_home, exp_away, home_team, away_team):
    """
    Scale lambdas away from the mean based on Elo gap.
    Strong-vs-weak matches get more extreme scorelines.
    """
    elo_h = ELO_RATINGS.get(home_team, GLOBAL_AVG_ELO)
    elo_a = ELO_RATINGS.get(away_team, GLOBAL_AVG_ELO)
    elo_gap = elo_h - elo_a   # positive = home stronger

    # multiplicative scale factor: winner gets boosted, loser gets suppressed
    scale = ELO_SCALE_K * elo_gap
    lam_h = exp_home * np.exp( scale)
    lam_a = exp_away * np.exp(-scale)

    return max(lam_h, 0.05), max(lam_a, 0.05)

In [23]:
def predict_wc_match(home_team, away_team, is_knockout=False):
    def raw_predict(ht, at):
        hf    = master.loc[ht]
        af    = master.loc[at]
        elo_h = ELO_RATINGS.get(ht, GLOBAL_AVG_ELO)
        elo_a = ELO_RATINGS.get(at, GLOBAL_AVG_ELO)

        feat = np.array([[
            elo_h - elo_a,
            elo_h,
            elo_a,
            hf['avg_goals_for']     - af['avg_goals_for'],
            hf['avg_goals_against'] - af['avg_goals_against'],
            hf['avg_points']        - af['avg_points'],
            hf['avg_goal_diff']     - af['avg_goal_diff'],
            hf['win_rate']          - af['win_rate'],
            hf['clean_sheet_rate']  - af['clean_sheet_rate'],
            hf['avg_goals_for'],
            hf['avg_goals_against'],
            hf['avg_points'],
            hf['win_rate'],
            af['avg_goals_for'],
            af['avg_goals_against'],
            af['avg_points'],
            af['win_rate'],
            1,          # is_neutral — always 1 for WC, bias cancelled by averaging
            1,          # is_wc
            int(is_knockout),
            get_squad_value(ht),
            get_squad_value(at),
            np.log(max(get_squad_value(ht), 1) / max(get_squad_value(at), 1))
        ]])
        return (float(home_model.predict(feat)[0]),
                float(away_model.predict(feat)[0]))

    fwd_h, fwd_a = raw_predict(home_team, away_team)
    flp_a, flp_h = raw_predict(away_team, home_team)

    exp_home = max(0, (fwd_h + flp_h) / 2)
    exp_away = max(0, (fwd_a + flp_a) / 2)

    if home_team in WC_HOST_NATIONS:
        exp_home += home_advantage
    elif away_team in WC_HOST_NATIONS:
        exp_away += home_advantage

    exp_home, exp_away = elo_scale_lambdas(exp_home, exp_away, home_team, away_team)

    return {
        'pred_home': int(round(exp_home)),
        'pred_away': int(round(exp_away)),
        'exp_home':  round(exp_home, 3),
        'exp_away':  round(exp_away, 3),
        'result':    'home' if int(round(exp_home)) > int(round(exp_away)) else
                     'away' if int(round(exp_away)) > int(round(exp_home)) else 'draw'
    }

In [24]:
def predict_knockout_match(home_team, away_team):
    """
    Knockout prediction:
    - XGBoost scoreline
    - If draw: run MC 1000 times, give +1 goal to higher win probability team
    - No penalties ever
    """
    pred = predict_wc_match(home_team, away_team, is_knockout=True)
    ph   = pred['pred_home']
    pa   = pred['pred_away']

    if ph > pa:
        return {
            'home_score': ph,
            'away_score': pa,
            'winner':     home_team,
            'result':     'home',
            'penalties':  False,
        }
    elif pa > ph:
        return {
            'home_score': ph,
            'away_score': pa,
            'winner':     away_team,
            'result':     'away',
            'penalties':  False,
        }
    else:
        # draw — use Poisson to get win probabilities
        mc = predict_match_mc(home_team, away_team, is_knockout=True)
        if mc['p_home'] >= mc['p_away']:
            winner = home_team
            result = 'home'
            ph     = ph + 1
        else:
            winner = away_team
            result = 'away'
            pa     = pa + 1
        return {
            'home_score': ph,
            'away_score': pa,
            'winner':     winner,
            'result':     result,
            'penalties':  False,
        }

In [25]:
# analytical poisson match probabilities
from scipy.stats import poisson as poisson_dist

MAX_GOALS = 10   # goals beyond this contribute < 0.01% probability

def poisson_win_prob(lam_h, lam_a):
    """
    analytically compute P(home win), P(draw), P(away win) and
    top-3 most probable scorelines from independent Poisson lambdas.
    """
    p_home, p_draw, p_away = 0.0, 0.0, 0.0
    scoreline_probs = {}

    for h in range(MAX_GOALS + 1):
        ph = poisson_dist.pmf(h, lam_h)
        for a in range(MAX_GOALS + 1):
            pa  = poisson_dist.pmf(a, lam_a)
            p   = ph * pa
            scoreline_probs[(h, a)] = p
            if h > a:
                p_home += p
            elif h == a:
                p_draw += p
            else:
                p_away += p

    top3 = sorted(scoreline_probs.items(), key=lambda x: x[1], reverse=True)[:3]

    # P(both teams score) = P(home >= 1) * P(away >= 1)
    p_btts = (1 - poisson_dist.pmf(0, lam_h)) * (1 - poisson_dist.pmf(0, lam_a))

    return {
        'p_home': round(p_home, 4),
        'p_draw': round(p_draw, 4),
        'p_away': round(p_away, 4),
        'p_btts': round(p_btts, 4),
        'top3':   [(f'{h}-{a}', round(p, 4)) for (h, a), p in top3],
    }


def predict_match_mc(home_team, away_team, is_knockout=False):
    """
    uses analytical Poisson.
    """
    pred = predict_wc_match(home_team, away_team, is_knockout=is_knockout)
    if pred is None:
        return None

    lam_h = max(pred['exp_home'], 0.05)
    lam_a = max(pred['exp_away'], 0.05)

    probs = poisson_win_prob(lam_h, lam_a)

    modal_h = int(round(lam_h))
    modal_a = int(round(lam_a))
    result  = 'home' if modal_h > modal_a else \
              'away' if modal_a > modal_h else 'draw'

    return {
        'home_team':  home_team,
        'away_team':  away_team,
        'modal_home': modal_h,
        'modal_away': modal_a,
        'exp_home':   round(lam_h, 4),
        'exp_away':   round(lam_a, 4),
        'p_home':     probs['p_home'],
        'p_draw':     probs['p_draw'],
        'p_away':     probs['p_away'],
        'p_btts':     probs['p_btts'],
        'top3':       probs['top3'],
        'result':     result,
    }

In [26]:
from collections import defaultdict, Counter
import itertools

# group definitions
# extracting from group_fixtures.csv
GROUPS = {}
for group_letter, gdf in group_fixtures.groupby('group'):
    teams_in_group = list(set(
        gdf['home_team'].tolist() + gdf['away_team'].tolist()
    ))
    GROUPS[group_letter] = teams_in_group

print("Groups:")
for g, teams in sorted(GROUPS.items()):
    print(f"  Group {g}: {teams}")

Groups:
  Group A: ['South Africa', 'Czech Republic', 'South Korea', 'Mexico']
  Group B: ['Qatar', 'Bosnia & Herzegovina', 'Canada', 'Switzerland']
  Group C: ['Haiti', 'Scotland', 'Brazil', 'Morocco']
  Group D: ['Turkey', 'Australia', 'USA', 'Paraguay']
  Group E: ['Curaçao', 'Germany', 'Ivory Coast', 'Ecuador']
  Group F: ['Netherlands', 'Sweden', 'Japan', 'Tunisia']
  Group G: ['Egypt', 'New Zealand', 'Iran', 'Belgium']
  Group H: ['Saudi Arabia', 'Cape Verde', 'Uruguay', 'Spain']
  Group I: ['France', 'Senegal', 'Iraq', 'Norway']
  Group J: ['Austria', 'Argentina', 'Jordan', 'Algeria']
  Group K: ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia']
  Group L: ['Ghana', 'England', 'Panama', 'Croatia']


In [27]:
def get_group_rankings(standings):
    """
    Sort group standings by: points → goal diff → goals for → wins.
    Returns ordered list of team names.
    """
    return sorted(
        standings.keys(),
        key=lambda t: (
            standings[t]['pts'],
            standings[t]['gd'],
            standings[t]['gf'],
            standings[t]['wins']),
        reverse=True)

In [28]:
# WC 2026 advancement rules
# 12 groups of 4:
# - Top 2 from each group advance automatically (24 teams)
# - Best 8 third-place teams also advance (8 teams)
# Total: 32 teams in Round of 32

def get_third_place_qualifiers(all_group_standings, all_rankings):
    """
    From all 12 groups, pick the 8 best third-place finishers.
    Ranked by: pts → gd → gf
    """
    third_place = []
    for group_letter, rankings in all_rankings.items():
        if len(rankings) >= 3:
            third_team = rankings[2]
            s = all_group_standings[group_letter][third_team]
            third_place.append({
                'team':  third_team,
                'group': group_letter,
                'pts':   s['pts'],
                'gd':    s['gd'],
                'gf':    s['gf'],
            })

    # sorting and taking best 8
    third_place.sort(key=lambda x: (x['pts'], x['gd'], x['gf']), reverse=True)
    return [t['team'] for t in third_place[:8]]

In [29]:
# knockout bracket simulator
# WC 2026 knockout bracket seeding follows FIFA's official bracket
# loading from knockout_slots.csv and resolving team slots from group results

knockout_slots = pd.read_csv('data/knockout_slots.csv')
knockout_slots['home_team'] = knockout_slots['slot_home'].copy()
knockout_slots['away_team'] = knockout_slots['slot_away'].copy()

## third place assignment

In [30]:
# third place assignment

def get_best_third_slot_assignments(third_qualifiers, all_rankings):
    third_with_groups = {}
    for g, rankings_list in all_rankings.items():
        if len(rankings_list) >= 3:
            third_team = rankings_list[2]
            if third_team in third_qualifiers:
                third_with_groups[third_team] = g

    best_third_slots = [
        ('Best 3rd (Groups A/B/C/D/F)',   {'A','B','C','D','F'}),
        ('Best 3rd (Groups C/D/F/G/H)',   {'C','D','F','G','H'}),
        ('Best 3rd (Groups C/E/F/H/I)',   {'C','E','F','H','I'}),
        ('Best 3rd (Groups E/H/I/J/K)',   {'E','H','I','J','K'}),
        ('Best 3rd (Groups A/E/H/I/J)',   {'A','E','H','I','J'}),
        ('Best 3rd (Groups B/E/F/I/J)',   {'B','E','F','I','J'}),
        ('Best 3rd (Groups E/F/G/I/J)',   {'E','F','G','I','J'}),
        ('Best 3rd (Groups D/E/I/J/L)',   {'D','E','I','J','L'}),
    ]

    slot_names  = [s[0] for s in best_third_slots]
    slot_groups = [s[1] for s in best_third_slots]
    teams       = list(third_with_groups.keys())

    slot_map   = {s: None for s in slot_names}
    used_teams = set()

    # pass 1 — backtracking for optimal valid assignment
    def backtrack(slot_idx):
        if slot_idx == len(slot_names):
            return True
        slot_name      = slot_names[slot_idx]
        slot_group_set = slot_groups[slot_idx]
        for team in teams:
            if team in used_teams:
                continue
            if third_with_groups[team] not in slot_group_set:
                continue
            slot_map[slot_name] = team
            used_teams.add(team)
            if backtrack(slot_idx + 1):
                return True
            slot_map[slot_name] = None
            used_teams.remove(team)
        return False

    backtrack(0)

    # pass 2 — force-assign any remaining unmatched teams to unfilled slots
    # this handles edge cases where perfect assignment is impossible
    unmatched_teams = [t for t in teams if t not in slot_map.values()]
    unfilled_slots  = [s for s in slot_names if slot_map[s] is None]

    for team, slot in zip(unmatched_teams, unfilled_slots):
        slot_map[slot] = team
        # no eligibility check — just fill it to prevent cascade failures

    return slot_map

# final predictions!! (xgb + monte carlo to resolve draws)

In [31]:
# FINAL PREDICTIONS — FULLY DETERMINISTIC
# Group stage + knockout bracket using XGBoost
# Draws in knockout resolved by analytical Poisson win probability

group_predictions = {}
ko_predictions    = {}

# group stage
print("=" * 65)
print("  GROUP STAGE PREDICTIONS")
print("=" * 65)

for group_letter in sorted(GROUPS.keys()):
    print(f"\n── Group {group_letter} " + "─" * 40)
    group_matches = group_fixtures[
        group_fixtures['group'] == group_letter
    ].sort_values('match_id')

    for _, match in group_matches.iterrows():
        mid = match['match_id']
        ht  = match['home_team']
        at  = match['away_team']

        pred = predict_match_mc(ht, at, is_knockout=False)
        ph   = pred['modal_home']
        pa   = pred['modal_away']

        if ph > pa:
            result = 'home'
            winner = ht
        elif pa > ph:
            result = 'away'
            winner = at
        else:
            result = 'draw'
            winner = None

        group_predictions[mid] = {
            'match_id':   mid,
            'group':      group_letter,
            'home_team':  ht,
            'away_team':  at,
            'home_score': ph,
            'away_score': pa,
            'result':     result,
            'winner':     winner,
            'exp_home':   pred['exp_home'],
            'exp_away':   pred['exp_away'],
            'p_home':     pred['p_home'],
            'p_draw':     pred['p_draw'],
            'p_away':     pred['p_away'],
            'p_btts':     pred['p_btts'],
            'top3':       pred['top3'],
        }

        winner_str = winner if winner else 'Draw'
        print(f"  Match {mid:>3}  {ht:22} {ph}-{pa}  "
              f"{at:22}  → {winner_str}")

# deterministic group standings
det_standings = {g: {t: {'pts':0,'gf':0,'ga':0,'gd':0,'wins':0}
                     for t in teams}
                 for g, teams in GROUPS.items()}

for mid, pred in group_predictions.items():
    g  = pred['group']
    ht = pred['home_team']
    at = pred['away_team']
    ph = pred['home_score']
    pa = pred['away_score']

    det_standings[g][ht]['gf'] += ph
    det_standings[g][ht]['ga'] += pa
    det_standings[g][ht]['gd'] += ph - pa
    det_standings[g][at]['gf'] += pa
    det_standings[g][at]['ga'] += ph
    det_standings[g][at]['gd'] += pa - ph

    if ph > pa:
        det_standings[g][ht]['pts']  += 3
        det_standings[g][ht]['wins'] += 1
    elif pa > ph:
        det_standings[g][at]['pts']  += 3
        det_standings[g][at]['wins'] += 1
    else:
        det_standings[g][ht]['pts'] += 1
        det_standings[g][at]['pts'] += 1

det_rankings = {g: get_group_rankings(s) for g, s in det_standings.items()}
det_winners  = {g: r[0] for g, r in det_rankings.items()}
det_runners  = {g: r[1] for g, r in det_rankings.items()}

# print standings
print(f"\n{'='*65}")
print(f"  PREDICTED GROUP STANDINGS")
print(f"{'='*65}")

for g in sorted(det_rankings.keys()):
    print(f"\n  Group {g}:")
    print(f"  {'Team':22} {'Pts':>4} {'GF':>4} {'GA':>4} {'GD':>4}")
    print(f"  {'-'*38}")
    for i, team in enumerate(det_rankings[g]):
        s      = det_standings[g][team]
        w      = s['wins']
        pts    = s['pts']
        d      = pts - (w * 3)
        l      = 3 - w - d
        marker = "✅" if i < 2 else "🔶" if team in get_third_place_qualifiers(
                     det_standings, det_rankings) else "❌"
        print(f"  {marker} {team:20} {s['pts']:>4} {s['gf']:>4} "
              f"{s['ga']:>4} {s['gd']:>4}")

# third place qualifiers
det_third = get_third_place_qualifiers(det_standings, det_rankings)
print(f"\n  Best 3rd place qualifiers:")
for team in det_third:
    for g, r in det_rankings.items():
        if len(r) >= 3 and r[2] == team:
            s = det_standings[g][team]
            print(f"    {team:25} (Group {g})  "
                  f"Pts:{s['pts']}  GD:{s['gd']:+d}  GF:{s['gf']}")

# build bracket slot map
det_third_slots = get_best_third_slot_assignments(det_third, det_rankings)

team_in_slot = {}
for g, team in det_winners.items():
    team_in_slot[f'Winner Group {g}'] = team
for g, team in det_runners.items():
    team_in_slot[f'Runner-up Group {g}'] = team
for slot_name, team in det_third_slots.items():
    if team is not None:
        team_in_slot[slot_name] = team

# knockout stage
print(f"\n{'='*65}")
print(f"  KNOCKOUT STAGE PREDICTIONS")
print(f"{'='*65}")

rounds_order = ['Round of 32', 'Round of 16', 'Quarter-final',
                'Semi-final', 'Third-place playoff', 'Final']

for round_name in rounds_order:
    round_slots = knockout_slots[
        knockout_slots['round'] == round_name
    ].sort_values('match_id')

    if len(round_slots) == 0:
        continue

    print(f"\n── {round_name} " + "─" * (50 - len(round_name)))

    for _, slot in round_slots.iterrows():
        mid       = slot['match_id']
        home_slot = slot['slot_home']
        away_slot = slot['slot_away']

        home_team = team_in_slot.get(home_slot)
        away_team = team_in_slot.get(away_slot)

        if home_team is None or away_team is None:
            print(f"  Match {mid:>3}  ⚠️  "
                  f"Could not resolve: '{home_slot}' vs '{away_slot}'")
            continue

        pred   = predict_knockout_match(home_team, away_team)
        ph     = pred['home_score']
        pa     = pred['away_score']
        winner = pred['winner']
        loser  = away_team if winner == home_team else home_team

        # storing analytical probs for the summary report
        mc = predict_match_mc(home_team, away_team, is_knockout=True)

        ko_predictions[mid] = {
            'match_id':   mid,
            'round':      round_name,
            'home_team':  home_team,
            'away_team':  away_team,
            'home_score': ph,
            'away_score': pa,
            'result':     pred['result'],
            'winner':     winner,
            'penalties':  False,
            'exp_home':   mc['exp_home'],
            'exp_away':   mc['exp_away'],
            'p_home':     mc['p_home'],
            'p_draw':     mc['p_draw'],
            'p_away':     mc['p_away'],
            'p_btts':     mc['p_btts'],
            'top3':       mc['top3'],
        }

        team_in_slot[f'Winner Match {mid}'] = winner
        team_in_slot[f'Loser Match {mid}']  = loser

        print(f"  Match {mid:>3}  {home_team:22} {ph}-{pa}  "
              f"{away_team:22}  → {winner}")

# final result
print(f"\n{'='*65}")
final_mid = knockout_slots[
    knockout_slots['round'] == 'Final'
]['match_id'].values[0]
if final_mid in ko_predictions:
    f = ko_predictions[final_mid]
    print(f"  PREDICTED CHAMPION: {f['winner']}")
    print(f"  FINAL: {f['home_team']} {f['home_score']}-"
          f"{f['away_score']} {f['away_team']}")
print(f"{'='*65}")

  GROUP STAGE PREDICTIONS

── Group A ────────────────────────────────────────
  Match   1  Mexico                 3-1  South Africa            → Mexico
  Match   2  South Korea            1-1  Czech Republic          → Draw
  Match  25  Czech Republic         2-1  South Africa            → Czech Republic
  Match  28  Mexico                 2-1  South Korea             → Mexico
  Match  53  Czech Republic         1-2  Mexico                  → Mexico
  Match  54  South Africa           1-2  South Korea             → South Korea

── Group B ────────────────────────────────────────
  Match   3  Canada                 2-1  Bosnia & Herzegovina    → Canada
  Match   6  Qatar                  0-3  Switzerland             → Switzerland
  Match  26  Switzerland            3-1  Bosnia & Herzegovina    → Switzerland
  Match  27  Canada                 3-1  Qatar                   → Canada
  Match  49  Switzerland            1-1  Canada                  → Draw
  Match  50  Bosnia & Herzegovina  